In [1]:
import numpy as np
import pandas as pd
from datasets import *

In [2]:
def compute_conditional_correlation(Sigma, i, k):
    """
    计算协方差矩阵 Sigma 中母线 i 和母线 k 的条件相关系数。
    
    参数:
      Sigma : np.ndarray
          整个系统的协方差矩阵
      i, k : int
          待计算的两个母线索引
      
    返回:
      correlation : float
          母线 i 和母线 k 的条件相关系数，计算公式为：
          rho_ik = Sigma_cond(0,1) / sqrt(Sigma_cond(0,0)*Sigma_cond(1,1))
    """
    n = Sigma.shape[0]
    # 定义索引集合：目标母线 i 和 k
    indices_I = [i, k]
    # 剩余母线索引集合
    indices_K = [j for j in range(n) if j not in indices_I]
    
    # 提取子矩阵
    Sigma_II = Sigma[np.ix_(indices_I, indices_I)]
    Sigma_IK = Sigma[np.ix_(indices_I, indices_K)]
    Sigma_KK = Sigma[np.ix_(indices_K, indices_K)]
    
    # 计算条件协方差矩阵（Schur 补）
    Sigma_KK_inv = np.linalg.inv(Sigma_KK)
    Sigma_cond = Sigma_II - Sigma_IK @ Sigma_KK_inv @ Sigma_IK.T
    
    # 计算条件相关系数
    correlation = Sigma_cond[0, 1] / np.sqrt(Sigma_cond[0, 0] * Sigma_cond[1, 1])
    return correlation

def detect_outage(Sigma_pre, Sigma_post, i, k, delta_max=0.4, delta_min=0.15):
    """
    检测母线 i 和母线 k 之间的支路是否发生断线。
    
    参数:
      Sigma_pre : np.ndarray
          正常状态下（停电前）的协方差矩阵
      Sigma_post : np.ndarray
          停电状态下的协方差矩阵
      i, k : int
          待检测支路对应的两个母线索引
      delta_max : float
          停电前条件相关系数的阈值下限（默认0.5）
      delta_min : float
          停电后条件相关系数的阈值上限（默认0.1）
    
    返回:
      outage : bool
          如果支路断线，则返回 True，否则返回 False
      rho_minus : float
          停电前计算得到的条件相关系数
      rho_plus : float
          停电后计算得到的条件相关系数
    """
    # 计算正常状态和停电状态下的条件相关系数
    rho_minus = compute_conditional_correlation(Sigma_pre, i, k)
    rho_plus = compute_conditional_correlation(Sigma_post, i, k)
    
    # 根据阈值判断支路是否断线
    outage = (np.abs(rho_minus) > delta_max) and (np.abs(rho_plus) < delta_min)
    return outage, rho_minus, rho_plus

In [3]:
dataset = BinaryDataset('Node123_loop')
n_buses = dataset.n_buses

df_normal = dataset.data_df
df_outage = dataset.data_outage_df

In [4]:
# 计算协方差矩阵，注意：pandas 的 cov() 默认按列计算协方差
Sigma_pre = df_normal.cov().to_numpy()
Sigma_post = df_outage.cov().to_numpy()

In [5]:
outage_results = []

In [6]:
# 遍历所有母线对（只考虑一次组合）
for i in range(n_buses):
    for k in range(i+1, n_buses):
        outage, rho_minus, rho_plus = detect_outage(Sigma_pre, Sigma_post, i, k)
        if outage:
            outage_results.append((i, k, rho_minus, rho_plus))

/tmp/ipykernel_138907/1453498425.py:32: RuntimeWarning: invalid value encountered in sqrt
  correlation = Sigma_cond[0, 1] / np.sqrt(Sigma_cond[0, 0] * Sigma_cond[1, 1])


In [7]:
# 输出检测结果
if outage_results:
    print("Outage detected：")
    for (i, k, rho_minus, rho_plus) in outage_results:
        print(f"Bus {i+2} and Bus {k+2}: Normal cov = {rho_minus:.3f}, outage cov = {rho_plus:.3f}")
else:
    print("No outage detected.")

Outage detected：
Bus 3 and Bus 59: Normal cov = -0.400, outage cov = -0.009
Bus 3 and Bus 101: Normal cov = -0.757, outage cov = 0.085
Bus 4 and Bus 30: Normal cov = -0.427, outage cov = -0.101
Bus 4 and Bus 40: Normal cov = -0.519, outage cov = -0.090
Bus 4 and Bus 55: Normal cov = -0.433, outage cov = -0.054
Bus 4 and Bus 84: Normal cov = 0.845, outage cov = 0.006
Bus 5 and Bus 79: Normal cov = -1.194, outage cov = 0.059
Bus 6 and Bus 69: Normal cov = -0.456, outage cov = -0.074
Bus 6 and Bus 74: Normal cov = 0.491, outage cov = 0.047
Bus 7 and Bus 18: Normal cov = -4.638, outage cov = -0.136
Bus 7 and Bus 74: Normal cov = 0.426, outage cov = -0.028
Bus 7 and Bus 110: Normal cov = -0.771, outage cov = -0.137
Bus 7 and Bus 118: Normal cov = -0.450, outage cov = -0.145
Bus 7 and Bus 122: Normal cov = -0.762, outage cov = -0.111
Bus 9 and Bus 21: Normal cov = -0.598, outage cov = 0.141
Bus 10 and Bus 12: Normal cov = 0.506, outage cov = 0.074
Bus 10 and Bus 20: Normal cov = -0.631, outa